# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the [FAIR²](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
Croissant schema URL: https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and examine dataset properties using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Show summary metadata
meta = dataset.metadata
print(f"Name: {meta.name}")
print(f"Version: {meta.version}")
print(f"Description: {meta.description}")
print(f"Published: {meta.datePublished}")

## 2. Data Overview
Inspect available **record sets**, **fields**, and their `@id` values, as described in the Croissant schema.

We will programmatically explore:
- All record sets and their `@id` and names
- For each record set, list all associated fields and columns by their `@id` and names

In [ ]:
# List all record sets and their fields
record_sets = dataset.record_sets
print(f"Found {len(record_sets)} record set(s):\n")

for rs in record_sets:
    print(f"- RecordSet: {rs.name} (id: {rs.id})")
    if hasattr(rs, 'fields') and rs.fields:
        print("  Fields:")
        for field in rs.fields:
            print(f"    * {field.name} (id: {field.id}) type: {field.data_type}")
    if hasattr(rs, 'columns') and rs.columns:
        print("  Columns:")
        for col in rs.columns:
            print(f"    * {col.name} (id: {col.id})")
    print('-' * 50)

# Save the first record set @id for next steps
if len(record_sets) > 0:
    example_rs_id = record_sets[0].id

## 3. Data Extraction
Load tabular data from each available record set into a DataFrame.

**All references to fields, columns, or record sets use their `@id` values as required by the Croissant specification.**

In [ ]:
# Load all dataframes referenced by record sets
dataframes = {}
record_set_ids = [rs.id for rs in dataset.record_sets]

for rsid in record_set_ids:
    records_iter = dataset.records(record_set=rsid)
    # Convert to DataFrame
    records_list = list(records_iter)
    if records_list:
        dataframes[rsid] = pd.DataFrame(records_list)
        print(f"Loaded {len(dataframes[rsid])} records for RecordSet id: {rsid}")

# Example: Show columns in the primary record set
if record_set_ids:
    main_rs_id = record_set_ids[0]
    print(f"Columns in primary record set (id: {main_rs_id}):")
    print(dataframes[main_rs_id].columns.tolist())
    display(dataframes[main_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
### Example Operations:

- Filtering rows by a numeric field
- Normalizing a numeric column
- Grouping by a categorical field

Choose numeric and grouping field `@id`s from the overview above for demonstration. Adjust field IDs if available.

In [ ]:
# --- Replace field ID with actual IDs discovered above if desired ---
record_set_id = main_rs_id
# Example field names (these should be replaced by their @id as per schema; demo assumes a field named 'MW' for molecular weight, and 'accession' as categorical)

# Try common field IDs
numeric_field_id = None
group_field_id = None

# Select first numeric field (guess by typical column names)
candidates = ['MW', 'molecular_weight', 'coverage', 'peptide_counts', 'SequenceCoverage_percentage']
if record_set_id in dataframes:
    df = dataframes[record_set_id]
    for c in candidates:
        c_id = c
        if c_id in df.columns:
            numeric_field_id = c_id
            break
    # Try first non-numeric as group
    group_candidates = ['accession', 'id', 'protein_id', 'description', 'Sample']
    for g in group_candidates:
        if g in df.columns:
            group_field_id = g
            break

# Fallback to any numeric column
if numeric_field_id is None and record_set_id in dataframes:
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break

print(f"Using record set: {record_set_id}\nNumeric field: {numeric_field_id}\nGroup field: {group_field_id}")

if numeric_field_id and numeric_field_id in df.columns:
    threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 0
    print(f"Threshold set to mean: {threshold}")

    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered {len(filtered_df)} records with {numeric_field_id} > {threshold}")
    display(filtered_df[[numeric_field_id]].head())

    filtered_df[numeric_field_id + '_normalized'] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id}:")
    display(filtered_df[[numeric_field_id, numeric_field_id + '_normalized']].head())

    # Group by a categorical field and aggregate mean
    if group_field_id and group_field_id in filtered_df.columns:
        grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame('mean_' + numeric_field_id)
        print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
        display(grouped.head())

## 5. Visualization
Visualize the distribution of the selected numeric field, and (optionally) relationships with the group field.

In [ ]:
import matplotlib.pyplot as plt

if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(8, 4))
    df[numeric_field_id].hist(bins=30)
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.title(f'Distribution of {numeric_field_id}')
    plt.show()

    # If grouping field is categorical with small unique count, show boxplot
    if group_field_id and group_field_id in df.columns:
        n_unique = df[group_field_id].nunique()
        if n_unique < 20:
            plt.figure(figsize=(12, 6))
            df.boxplot(column=numeric_field_id, by=group_field_id)
            plt.title(f"{numeric_field_id} by {group_field_id}")
            plt.suptitle("")
            plt.xlabel(group_field_id)
            plt.ylabel(numeric_field_id)
            plt.show()

## 6. Conclusion

- We explored the FAIR² dataset's core record sets, fields, and performed field-driven analysis using the `mlcroissant` library.
- Data was filtered and normalized, and basic summaries and visualizations demonstrated the usage of `mlcroissant` with tabular data.

**Next steps:** Use the dataframes and their reference `@id`s for advanced analysis, ML modeling, or additional FAIR operations as needed.